<a href="https://colab.research.google.com/github/elizameloquintana-debug/Trabajo-fin-Python/blob/main/tutorial_api_completo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tutorial completo: construcción de API financiera paso a paso

Este notebook te guiará a través de la creación de una API REST financiera desde cero, explicando cada función y su propósito.

## Objetivos de aprendizaje

- Comprender la arquitectura de una API REST
- Crear modelos de datos con SQLAlchemy
- Implementar endpoints RESTful
- Aplicar autenticación y validación
- Integrar con herramientas de automatización

## Índice

1. [Fundamentos y arquitectura](#1.-Fundamentos-y-arquitectura)
2. [Modelos de datos](#2.-Modelos-de-datos)
3. [Rutas y endpoints](#3.-Rutas-y-endpoints)
4. [Autenticación](#4.-Autenticación)
5. [Integración práctica](#5.-Integración-práctica)
6. [Testing](#6.-Testing)

## 1. Fundamentos y arquitectura

### ¿Qué es una API REST?

Una API REST (Representational State Transfer) es una interfaz que permite que diferentes aplicaciones se comuniquen mediante HTTP.

### Arquitectura de nuestra API

```
┌─────────────────────────────────────────┐
│  Cliente (n8n, Make, Zapier, Browser)   │
└────────────────┬────────────────────────┘
                 │ HTTP Request (JSON)
                 ▼
┌─────────────────────────────────────────┐
│         Flask Application (app.py)      │
│  ┌───────────────────────────────────┐  │
│  │  Middleware (Autenticación)       │  │
│  └───────────────────────────────────┘  │
│  ┌───────────────────────────────────┐  │
│  │  Routes (routes.py)               │  │
│  │  - /api/customers                 │  │
│  │  - /api/transactions              │  │
│  │  - /api/reports                   │  │
│  └───────────────────────────────────┘  │
│  ┌───────────────────────────────────┐  │
│  │  Models (models.py)               │  │
│  │  - Customer                       │  │
│  │  - Transaction                    │  │
│  │  - AuditLog                       │  │
│  └───────────────────────────────────┘  │
└────────────────┬────────────────────────┘
                 │ SQL Queries
                 ▼
┌─────────────────────────────────────────┐
│     Base de Datos SQLite/PostgreSQL     │
└─────────────────────────────────────────┘
```

### Instalación de dependencias

Primero, instalamos las bibliotecas necesarias:

In [2]:
# Instalar dependencias (ejecutar solo una vez)
!pip install flask flask-sqlalchemy flask-cors python-dotenv requests

## 2. Modelos de datos

Los modelos definen la estructura de nuestras tablas en la base de datos.

### 2.1. Modelo Customer (Cliente)

**¿Por qué necesitamos este modelo?**
- Almacenar información de clientes
- Relacionar transacciones con clientes específicos
- Facilitar reportes por cliente

In [3]:
from flask_sqlalchemy import SQLAlchemy
from datetime import datetime

db = SQLAlchemy()

class Customer(db.Model):
    """Modelo de cliente

    Atributos:
        id: Identificador único del cliente
        email: Email único para contacto
        nombre: Nombre del cliente
        apellido: Apellido del cliente
        pais: País de residencia (código ISO)
        telefono: Número de contacto
        activo: Estado del cliente (True/False)
        created_at: Fecha de registro
        updated_at: Última actualización

    Relaciones:
        transacciones: Lista de transacciones del cliente
    """
    __tablename__ = 'customers'

    # Columnas de la tabla
    id = db.Column(db.Integer, primary_key=True)
    email = db.Column(db.String(120), unique=True, nullable=False)
    nombre = db.Column(db.String(100), nullable=False)
    apellido = db.Column(db.String(100))
    pais = db.Column(db.String(50))
    telefono = db.Column(db.String(20))
    activo = db.Column(db.Boolean, default=True)
    created_at = db.Column(db.DateTime, default=datetime.utcnow)
    updated_at = db.Column(db.DateTime, default=datetime.utcnow, onupdate=datetime.utcnow)

    # Relación con transacciones (un cliente puede tener muchas transacciones)
    transacciones = db.relationship('Transaction', backref='customer', lazy=True)

    def to_dict(self):
        """Convierte el objeto a diccionario para serialización JSON

        ¿Por qué esta función?
        - SQLAlchemy devuelve objetos, no diccionarios
        - Las APIs REST necesitan devolver JSON
        - Permite controlar qué campos exponer
        """
        return {
            'id': self.id,
            'email': self.email,
            'nombre': self.nombre,
            'apellido': self.apellido,
            'nombre_completo': f"{self.nombre} {self.apellido or ''}".strip(),
            'pais': self.pais,
            'telefono': self.telefono,
            'activo': self.activo,
            'created_at': self.created_at.isoformat() if self.created_at else None,
            'total_transacciones': len(self.transacciones)
        }

print("✓ Modelo Customer definido correctamente")

✓ Modelo Customer definido correctamente


### 2.2. Modelo Transaction (Transacción)

**¿Por qué necesitamos este modelo?**
- Registrar todas las transacciones financieras
- Rastrear estados (pendiente, completada, fallida)
- Permitir auditoría y análisis
- Soportar múltiples monedas y tipos

In [4]:
class Transaction(db.Model):
    """Modelo de transacción financiera

    Atributos clave:
        customer_id: FK a Customer (relación)
        monto: Cantidad numérica (10 dígitos, 2 decimales)
        moneda: Código ISO 4217 (USD, EUR, etc.)
        tipo: payment, refund, transfer
        estado: pending, completed, failed, cancelled
        referencia_externa: ID de sistema externo (Stripe, PayPal)
        datos_metadata: JSON para datos adicionales flexibles
    """
    __tablename__ = 'transactions'

    id = db.Column(db.Integer, primary_key=True)
    customer_id = db.Column(db.Integer, db.ForeignKey('customers.id'), nullable=False)

    # Campos financieros
    monto = db.Column(db.Numeric(10, 2), nullable=False)
    moneda = db.Column(db.String(3), default='USD')

    # Campos de estado
    tipo = db.Column(db.String(20), nullable=False)
    estado = db.Column(db.String(20), default='pending')

    # Información adicional
    descripcion = db.Column(db.String(255))
    referencia_externa = db.Column(db.String(100))

    # Fechas
    fecha = db.Column(db.DateTime, default=datetime.utcnow)
    fecha_completado = db.Column(db.DateTime)

    # Metadata flexible (JSON) - ¡Renombrado para evitar conflicto con SQLAlchemy!
    datos_metadata = db.Column(db.JSON)

    created_at = db.Column(db.DateTime, default=datetime.utcnow)
    updated_at = db.Column(db.DateTime, default=datetime.utcnow, onupdate=datetime.utcnow)

    # Relación con auditoría
    audit_logs = db.relationship('AuditLog', backref='transaction', lazy=True)

    def to_dict(self):
        """Serialización a JSON

        Incluye datos relacionados del cliente para evitar
        múltiples queries (optimización N+1)
        """
        return {
            'id': self.id,
            'customer_id': self.customer_id,
            'customer_email': self.customer.email if self.customer else None,
            'customer_nombre': self.customer.nombre if self.customer else None,
            'monto': float(self.monto),
            'moneda': self.moneda,
            'tipo': self.tipo,
            'estado': self.estado,
            'descripcion': self.descripcion,
            'referencia_externa': self.referencia_externa,
            'fecha': self.fecha.isoformat() if self.fecha else None,
            'fecha_completado': self.fecha_completado.isoformat() if self.fecha_completado else None,
            'metadata': self.datos_metadata, # Actualizado aquí también para serializarlo correctamente
            'created_at': self.created_at.isoformat() if self.created_at else None
        }

print("✓ Modelo Transaction definido correctamente")

✓ Modelo Transaction definido correctamente


### 2.3. Modelo AuditLog (Registro de auditoría)

**¿Por qué necesitamos este modelo?**
- Cumplimiento regulatorio (GDPR, PCI-DSS)
- Rastrear quién hizo qué y cuándo
- Detectar fraude o errores
- Debugging de problemas

In [5]:
class AuditLog(db.Model):
    """Modelo de registro de auditoría

    Registra todos los cambios importantes en el sistema.

    Campos importantes:
        accion: created, updated, deleted
        usuario: Email o ID del usuario
        ip_address: IP desde donde se hizo la acción
        datos_anteriores: Estado antes del cambio (JSON)
        datos_nuevos: Estado después del cambio (JSON)
    """
    __tablename__ = 'audit_logs'

    id = db.Column(db.Integer, primary_key=True)
    transaction_id = db.Column(db.Integer, db.ForeignKey('transactions.id'))

    accion = db.Column(db.String(50), nullable=False)
    usuario = db.Column(db.String(100))
    ip_address = db.Column(db.String(50))

    datos_anteriores = db.Column(db.JSON)
    datos_nuevos = db.Column(db.JSON)

    timestamp = db.Column(db.DateTime, default=datetime.utcnow)

    def to_dict(self):
        return {
            'id': self.id,
            'transaction_id': self.transaction_id,
            'accion': self.accion,
            'usuario': self.usuario,
            'ip_address': self.ip_address,
            'timestamp': self.timestamp.isoformat() if self.timestamp else None,
            'datos_anteriores': self.datos_anteriores,
            'datos_nuevos': self.datos_nuevos
        }

print("✓ Modelo AuditLog definido correctamente")

✓ Modelo AuditLog definido correctamente


### 2.4. Función auxiliar para crear logs de auditoría

**¿Por qué esta función?**
- Evita código duplicado
- Garantiza consistencia en los logs
- Captura automáticamente IP y timestamp

In [6]:
def create_audit_log(transaction_id, accion, usuario='system', datos_anteriores=None, datos_nuevos=None):
    """Crear registro de auditoría

    Args:
        transaction_id: ID de la transacción afectada
        accion: Tipo de acción (created, updated, deleted)
        usuario: Identificador del usuario (default: 'system')
        datos_anteriores: Estado previo (dict)
        datos_nuevos: Estado nuevo (dict)

    Returns:
        AuditLog: Objeto de auditoría creado

    Ejemplo de uso:
        log = create_audit_log(
            transaction_id=123,
            accion='updated',
            usuario='admin@empresa.com',
            datos_anteriores={'estado': 'pending'},
            datos_nuevos={'estado': 'completed'}
        )
        db.session.add(log)
        db.session.commit()
    """
    from flask import request

    log = AuditLog(
        transaction_id=transaction_id,
        accion=accion,
        usuario=usuario,
        ip_address=request.remote_addr if request else None,
        datos_anteriores=datos_anteriores,
        datos_nuevos=datos_nuevos
    )

    db.session.add(log)
    return log

print("✓ Función create_audit_log definida")

✓ Función create_audit_log definida


## 3. Rutas y endpoints

Las rutas definen los endpoints de nuestra API.

### 3.1. Endpoint GET /api/customers

**¿Por qué esta función?**
- Listar todos los clientes
- Soportar filtros (activo, país)
- Implementar paginación para grandes datasets

In [7]:
from flask import Flask, jsonify, request

app = Flask(__name__)

@app.route('/api/customers', methods=['GET'])
def get_customers():
    """Listar clientes con filtros y paginación

    Query Parameters:
        activo (bool): Filtrar por estado activo
        pais (str): Filtrar por país (código ISO)
        page (int): Número de página (default: 1)
        per_page (int): Elementos por página (default: 50)

    Returns:
        JSON con:
        - success: True/False
        - data: Array de clientes
        - pagination: Información de paginación

    Ejemplo:
        GET /api/customers?activo=true&pais=ES&page=1&per_page=10
    """
    try:
        # Obtener parámetros de filtro
        activo = request.args.get('activo', type=bool)
        pais = request.args.get('pais')

        # Query base
        query = Customer.query

        # Aplicar filtros si existen
        if activo is not None:
            query = query.filter_by(activo=activo)

        if pais:
            query = query.filter_by(pais=pais)

        # Paginación
        page = request.args.get('page', 1, type=int)
        per_page = request.args.get('per_page', 50, type=int)

        # Ejecutar query con paginación
        customers = query.paginate(page=page, per_page=per_page, error_out=False)

        # Construir respuesta
        return jsonify({
            'success': True,
            'data': [c.to_dict() for c in customers.items],
            'pagination': {
                'page': page,
                'per_page': per_page,
                'total': customers.total,
                'pages': customers.pages
            }
        })

    except Exception as e:
        return jsonify({'error': str(e)}), 500

print("✓ Endpoint GET /api/customers definido")

✓ Endpoint GET /api/customers definido


### 3.2. Endpoint POST /api/customers

**¿Por qué esta función?**
- Crear nuevos clientes
- Validar datos de entrada
- Garantizar unicidad de email

In [8]:
@app.route('/api/customers', methods=['POST'])
def create_customer():
    """Crear nuevo cliente

    Request Body (JSON):
        {
            "email": "cliente@ejemplo.com",  # Requerido
            "nombre": "Juan",                # Requerido
            "apellido": "Pérez",             # Opcional
            "pais": "ES",                    # Opcional (default: ES)
            "telefono": "+34600111222"       # Opcional
        }

    Validaciones:
        - Email es requerido y debe ser único
        - Nombre es requerido
        - Email debe tener formato válido (implementación futura)

    Returns:
        201: Cliente creado exitosamente
        400: Datos inválidos o email duplicado
        500: Error interno
    """
    try:
        data = request.get_json()

        # Validación: email requerido
        if not data.get('email'):
            return jsonify({'error': 'Email es requerido'}), 400

        # Validación: nombre requerido
        if not data.get('nombre'):
            return jsonify({'error': 'Nombre es requerido'}), 400

        # Validación: email único
        if Customer.query.filter_by(email=data['email']).first():
            return jsonify({'error': 'Email ya existe'}), 400

        # Crear cliente
        customer = Customer(
            email=data['email'],
            nombre=data['nombre'],
            apellido=data.get('apellido'),
            pais=data.get('pais', 'ES'),
            telefono=data.get('telefono')
        )

        # Guardar en base de datos
        db.session.add(customer)
        db.session.commit()

        return jsonify({
            'success': True,
            'message': 'Cliente creado exitosamente',
            'data': customer.to_dict()
        }), 201

    except Exception as e:
        db.session.rollback()
        return jsonify({'error': str(e)}), 500

print("✓ Endpoint POST /api/customers definido")

✓ Endpoint POST /api/customers definido


### 3.3. Endpoint GET /api/transactions

**¿Por qué esta función?**
- Obtener transacciones con múltiples filtros
- Soportar búsquedas complejas (fecha, monto, estado)
- Ordenar resultados
- Optimizar para grandes volúmenes de datos

In [9]:
@app.route('/api/transactions', methods=['GET'])
def get_transactions():
    """Listar transacciones con filtros avanzados

    Query Parameters:
        customer_id (int): Filtrar por cliente
        tipo (str): payment, refund, transfer
        estado (str): pending, completed, failed, cancelled
        moneda (str): USD, EUR, GBP, etc.
        fecha_desde (ISO): Fecha inicio (YYYY-MM-DD)
        fecha_hasta (ISO): Fecha fin (YYYY-MM-DD)
        monto_min (float): Monto mínimo
        monto_max (float): Monto máximo
        order_by (str): Campo para ordenar (default: fecha)
        order_dir (str): asc o desc (default: desc)
        page (int): Número de página
        per_page (int): Elementos por página

    Ejemplo:
        GET /api/transactions?estado=completed&monto_min=100&order_by=monto&order_dir=desc
    """
    try:
        # Obtener filtros
        customer_id = request.args.get('customer_id', type=int)
        tipo = request.args.get('tipo')
        estado = request.args.get('estado')
        moneda = request.args.get('moneda')

        # Filtros de fecha
        fecha_desde = request.args.get('fecha_desde')
        fecha_hasta = request.args.get('fecha_hasta')

        # Filtros de monto
        monto_min = request.args.get('monto_min', type=float)
        monto_max = request.args.get('monto_max', type=float)

        # Query base
        query = Transaction.query

        # Aplicar filtros simples
        if customer_id:
            query = query.filter_by(customer_id=customer_id)

        if tipo:
            query = query.filter_by(tipo=tipo)

        if estado:
            query = query.filter_by(estado=estado)

        if moneda:
            query = query.filter_by(moneda=moneda)

        # Filtros de fecha (requieren conversión)
        if fecha_desde:
            query = query.filter(Transaction.fecha >= datetime.fromisoformat(fecha_desde))

        if fecha_hasta:
            query = query.filter(Transaction.fecha <= datetime.fromisoformat(fecha_hasta))

        # Filtros de monto
        if monto_min:
            query = query.filter(Transaction.monto >= monto_min)

        if monto_max:
            query = query.filter(Transaction.monto <= monto_max)

        # Ordenamiento
        order_by = request.args.get('order_by', 'fecha')
        order_dir = request.args.get('order_dir', 'desc')

        if order_dir == 'desc':
            query = query.order_by(getattr(Transaction, order_by).desc())
        else:
            query = query.order_by(getattr(Transaction, order_by).asc())

        # Paginación
        page = request.args.get('page', 1, type=int)
        per_page = request.args.get('per_page', 50, type=int)

        transactions = query.paginate(page=page, per_page=per_page, error_out=False)

        return jsonify({
            'success': True,
            'data': [t.to_dict() for t in transactions.items],
            'pagination': {
                'page': page,
                'per_page': per_page,
                'total': transactions.total,
                'pages': transactions.pages
            }
        })

    except Exception as e:
        return jsonify({'error': str(e)}), 500

print("✓ Endpoint GET /api/transactions definido")

✓ Endpoint GET /api/transactions definido


### 3.4. Endpoint POST /api/transactions

**¿Por qué esta función?**
- Crear nuevas transacciones
- Validar datos financieros
- Registrar automáticamente en auditoría
- Verificar existencia del cliente

In [10]:
@app.route('/api/transactions', methods=['POST'])
def create_transaction():
    """Crear nueva transacción

    Request Body:
        {
            "customer_id": 1,              # Requerido
            "monto": 99.99,                # Requerido
            "moneda": "EUR",               # Opcional (default: USD)
            "tipo": "payment",             # Requerido
            "estado": "pending",           # Opcional (default: pending)
            "descripcion": "...",          # Opcional
            "referencia_externa": "...",   # Opcional
            "metadata": {}                 # Opcional (JSON)
        }

    Validaciones:
        - customer_id debe existir
        - monto debe ser número positivo
        - tipo debe ser: payment, refund, transfer

    Side Effects:
        - Crea registro de auditoría automáticamente
    """
    try:
        data = request.get_json()

        # Validaciones
        if not data.get('customer_id'):
            return jsonify({'error': 'customer_id es requerido'}), 400

        if not data.get('monto'):
            return jsonify({'error': 'monto es requerido'}), 400

        if not data.get('tipo'):
            return jsonify({'error': 'tipo es requerido'}), 400

        # Verificar que el cliente existe
        customer = Customer.query.get(data['customer_id'])
        if not customer:
            return jsonify({'error': 'Cliente no encontrado'}), 404

        # Crear transacción
        transaction = Transaction(
            customer_id=data['customer_id'],
            monto=data['monto'],
            moneda=data.get('moneda', 'USD'),
            tipo=data['tipo'],
            estado=data.get('estado', 'pending'),
            descripcion=data.get('descripcion'),
            referencia_externa=data.get('referencia_externa'),
            metadata=data.get('metadata')
        )

        db.session.add(transaction)
        db.session.commit()

        # Crear log de auditoría
        create_audit_log(
            transaction_id=transaction.id,
            accion='created',
            datos_nuevos=transaction.to_dict()
        )
        db.session.commit()

        return jsonify({
            'success': True,
            'message': 'Transacción creada exitosamente',
            'data': transaction.to_dict()
        }), 201

    except Exception as e:
        db.session.rollback()
        return jsonify({'error': str(e)}), 500

print("✓ Endpoint POST /api/transactions definido")

✓ Endpoint POST /api/transactions definido


### 3.5. Endpoint GET /api/reports/daily

**¿Por qué esta función?**
- Generar reportes analíticos
- Calcular métricas agregadas
- Facilitar dashboards y alertas
- Optimizar para análisis (vs transacciones individuales)

In [11]:
from sqlalchemy import func

@app.route('/api/reports/daily', methods=['GET'])
def daily_report():
    """Generar reporte diario de transacciones

    Query Parameters:
        fecha (ISO): Fecha del reporte (default: hoy)

    Returns:
        {
            "fecha": "2024-03-27",
            "total_transacciones": 45,
            "monto_total": 12345.67,
            "monto_promedio": 274.35,
            "transaccion_minima": 10.00,
            "transaccion_maxima": 2500.00,
            "por_tipo": {
                "payment": {"count": 40, "total": 11000},
                "refund": {"count": 5, "total": 1345.67}
            },
            "por_estado": {
                "completed": 42,
                "pending": 2,
                "failed": 1
            },
            "clientes_unicos": 28
        }

    Uso típico:
        - Reportes diarios automatizados (email, Slack)
        - Dashboards de métricas
        - Alertas de anomalías
    """
    try:
        # Fecha objetivo (hoy por defecto)
        fecha_str = request.args.get('fecha')
        if fecha_str:
            fecha = datetime.fromisoformat(fecha_str).date()
        else:
            fecha = datetime.utcnow().date()

        # Inicio y fin del día
        inicio_dia = datetime.combine(fecha, datetime.min.time())
        fin_dia = datetime.combine(fecha, datetime.max.time())

        # Obtener transacciones del día
        transacciones = Transaction.query.filter(
            Transaction.fecha >= inicio_dia,
            Transaction.fecha <= fin_dia
        ).all()

        # Calcular estadísticas
        total_transacciones = len(transacciones)

        if total_transacciones == 0:
            return jsonify({
                'success': True,
                'fecha': fecha.isoformat(),
                'total_transacciones': 0,
                'monto_total': 0,
                'monto_promedio': 0
            })

        montos = [float(t.monto) for t in transacciones]
        monto_total = sum(montos)
        monto_promedio = monto_total / total_transacciones

        # Agrupar por tipo
        por_tipo = {}
        for t in transacciones:
            if t.tipo not in por_tipo:
                por_tipo[t.tipo] = {'count': 0, 'total': 0}
            por_tipo[t.tipo]['count'] += 1
            por_tipo[t.tipo]['total'] += float(t.monto)

        # Agrupar por estado
        por_estado = {}
        for t in transacciones:
            if t.estado not in por_estado:
                por_estado[t.estado] = 0
            por_estado[t.estado] += 1

        return jsonify({
            'success': True,
            'fecha': fecha.isoformat(),
            'total_transacciones': total_transacciones,
            'monto_total': round(monto_total, 2),
            'monto_promedio': round(monto_promedio, 2),
            'transaccion_minima': round(min(montos), 2),
            'transaccion_maxima': round(max(montos), 2),
            'por_tipo': por_tipo,
            'por_estado': por_estado,
            'clientes_unicos': len(set(t.customer_id for t in transacciones))
        })

    except Exception as e:
        return jsonify({'error': str(e)}), 500

print("✓ Endpoint GET /api/reports/daily definido")

✓ Endpoint GET /api/reports/daily definido


## 4. Autenticación

### 4.1. Middleware de autenticación

**¿Por qué necesitamos autenticación?**
- Proteger datos sensibles
- Rastrear quién accede a qué
- Cumplir regulaciones (GDPR, PCI-DSS)
- Prevenir abuso de la API

In [12]:
import os

@app.before_request
def verify_api_key():
    """Middleware de autenticación mediante API Key

    Se ejecuta ANTES de cada petición.

    Endpoints públicos (sin autenticación):
        - /api/health
        - /

    Todos los demás endpoints requieren:
        Header: X-API-Key: valor_secreto

    ¿Por qué API Key y no OAuth?
        - Simplicidad para integraciones server-to-server
        - Adecuado para n8n, Make, Zapier
        - Fácil de implementar y testear

    Mejoras futuras:
        - Múltiples API keys con diferentes permisos
        - Rate limiting por API key
        - Expiración de keys
        - OAuth2 para clientes web
    """
    # Endpoints públicos (sin autenticación)
    public_endpoints = ['/api/health', '/']

    if request.path in public_endpoints:
        return None  # Continuar sin verificar

    # Obtener API key del header
    api_key = request.headers.get('X-API-Key')
    expected_key = os.getenv('API_KEY', 'api_key_demo_12345')

    # Verificar API key
    if api_key != expected_key:
        return jsonify({
            'error': 'No autorizado',
            'message': 'API key inválida o no proporcionada',
            'hint': 'Incluir header: X-API-Key'
        }), 401

print("✓ Middleware de autenticación definido")

✓ Middleware de autenticación definido


## 5. Integración práctica

### 5.1. Ejemplo: Cliente Python para la API

In [13]:
import requests
import json

class APIFinancieraClient:
    """Cliente Python para interactuar con la API Financiera

    Ejemplo de uso:
        client = APIFinancieraClient(
            base_url='http://localhost:5000/api',
            api_key='api_key_demo_12345'
        )

        # Obtener transacciones
        transacciones = client.get_transactions(estado='completed')

        # Crear nueva transacción
        nueva = client.create_transaction(
            customer_id=1,
            monto=99.99,
            moneda='EUR',
            tipo='payment'
        )
    """

    def __init__(self, base_url, api_key):
        self.base_url = base_url
        self.headers = {
            'X-API-Key': api_key,
            'Content-Type': 'application/json'
        }

    def get_transactions(self, **filters):
        """Obtener transacciones con filtros

        Args:
            **filters: customer_id, estado, tipo, moneda, etc.

        Returns:
            dict: Respuesta de la API
        """
        response = requests.get(
            f"{self.base_url}/transactions",
            headers=self.headers,
            params=filters
        )
        response.raise_for_status()
        return response.json()

    def create_transaction(self, customer_id, monto, tipo, **kwargs):
        """Crear nueva transacción

        Args:
            customer_id: ID del cliente
            monto: Cantidad
            tipo: payment, refund, transfer
            **kwargs: moneda, descripcion, etc.
        """
        data = {
            'customer_id': customer_id,
            'monto': monto,
            'tipo': tipo,
            **kwargs
        }

        response = requests.post(
            f"{self.base_url}/transactions",
            headers=self.headers,
            json=data
        )
        response.raise_for_status()
        return response.json()

    def get_daily_report(self, fecha=None):
        """Obtener reporte diario"""
        params = {'fecha': fecha} if fecha else {}

        response = requests.get(
            f"{self.base_url}/reports/daily",
            headers=self.headers,
            params=params
        )
        response.raise_for_status()
        return response.json()

# Ejemplo de uso
print("✓ Cliente Python definido")
print("\nEjemplo de uso:")
print("""
client = APIFinancieraClient(
    base_url='http://localhost:5000/api',
    api_key='api_key_demo_12345'
)

# Obtener transacciones completadas
transacciones = client.get_transactions(estado='completed')
print(f"Total: {len(transacciones['data'])} transacciones")

# Crear nueva transacción
nueva = client.create_transaction(
    customer_id=1,
    monto=150.00,
    moneda='EUR',
    tipo='payment',
    descripcion='Pago desde Python'
)
print(f"Transacción creada: ID {nueva['data']['id']}")

# Obtener reporte del día
reporte = client.get_daily_report()
print(f"Total del día: ${reporte['monto_total']}")
""")

✓ Cliente Python definido

Ejemplo de uso:

client = APIFinancieraClient(
    base_url='http://localhost:5000/api',
    api_key='api_key_demo_12345'
)

# Obtener transacciones completadas
transacciones = client.get_transactions(estado='completed')
print(f"Total: {len(transacciones['data'])} transacciones")

# Crear nueva transacción
nueva = client.create_transaction(
    customer_id=1,
    monto=150.00,
    moneda='EUR',
    tipo='payment',
    descripcion='Pago desde Python'
)
print(f"Transacción creada: ID {nueva['data']['id']}")

# Obtener reporte del día
reporte = client.get_daily_report()
print(f"Total del día: ${reporte['monto_total']}")



### 5.2. Ejemplo: Integración con n8n

**Workflow n8n: Monitor de transacciones grandes**

In [14]:
# Este es un ejemplo del código JavaScript que usarías
# en un nodo Function de n8n

n8n_function_code = '''
// Nodo Function en n8n
// Filtrar transacciones mayores a $1000

const umbral = 1000;
const transacciones = items[0].json.data;

// Filtrar transacciones grandes
const grandes = transacciones.filter(tx => {
  return parseFloat(tx.monto) > umbral;
});

console.log(`Total transacciones: ${transacciones.length}`);
console.log(`Transacciones grandes (>${umbral}): ${grandes.length}`);

// Si no hay transacciones grandes, detener workflow
if (grandes.length === 0) {
  return [];
}

// Formatear para enviar alertas
return grandes.map(tx => ({
  json: {
    id: tx.id,
    monto: tx.monto,
    moneda: tx.moneda,
    cliente: tx.customer_nombre,
    email: tx.customer_email,
    alerta: `⚠️ Transacción grande: $${tx.monto} ${tx.moneda}`,
    timestamp: new Date().toISOString()
  }
}));
'''

print("✓ Ejemplo de código n8n")
print("\nEste código se usaría en un nodo Function de n8n para:")
print("1. Filtrar transacciones > $1000")
print("2. Formatear datos para alertas")
print("3. Pasar datos al siguiente nodo (Email, Slack, etc.)")

✓ Ejemplo de código n8n

Este código se usaría en un nodo Function de n8n para:
1. Filtrar transacciones > $1000
2. Formatear datos para alertas
3. Pasar datos al siguiente nodo (Email, Slack, etc.)


### 5.3. Ejemplo: Integración con Make

**Configuración de HTTP Request en Make:**

In [15]:
make_config = {
    "module": "HTTP - Make a request",
    "config": {
        "url": "http://tu-api.com/api/transactions",
        "method": "GET",
        "headers": [
            {
                "name": "X-API-Key",
                "value": "api_key_demo_12345"
            }
        ],
        "query_string": [
            {
                "name": "estado",
                "value": "completed"
            },
            {
                "name": "fecha_desde",
                "value": "{{addDays(now; -1)}}"
            }
        ],
        "parse_response": True
    }
}

print("✓ Configuración Make definida")
print("\nEsta configuración:")
print("1. Hace una petición GET a /api/transactions")
print("2. Incluye autenticación via API Key")
print("3. Filtra transacciones completadas del último día")
print("4. Parsea automáticamente la respuesta JSON")
print("\nLuego puedes usar {{module.data}} para acceder a los datos")

✓ Configuración Make definida

Esta configuración:
1. Hace una petición GET a /api/transactions
2. Incluye autenticación via API Key
3. Filtra transacciones completadas del último día
4. Parsea automáticamente la respuesta JSON

Luego puedes usar {{module.data}} para acceder a los datos


### 5.4. Ejemplo: Integración con Zapier

In [16]:
zapier_config = {
    "trigger": {
        "app": "Schedule by Zapier",
        "event": "Every Day",
        "time": "11:00 PM"
    },
    "action_1": {
        "app": "Webhooks by Zapier",
        "event": "GET",
        "config": {
            "url": "https://tu-api.com/api/reports/daily",
            "headers": {
                "X-API-Key": "api_key_demo_12345"
            }
        }
    },
    "action_2": {
        "app": "Google Sheets",
        "event": "Create Spreadsheet Row",
        "config": {
            "spreadsheet": "Reportes Diarios",
            "worksheet": "Transacciones",
            "columns": {
                "Fecha": "{{action_1.fecha}}",
                "Total Transacciones": "{{action_1.total_transacciones}}",
                "Monto Total": "{{action_1.monto_total}}",
                "Promedio": "{{action_1.monto_promedio}}"
            }
        }
    },
    "action_3": {
        "app": "Email by Zapier",
        "event": "Send Outbound Email",
        "config": {
            "to": "admin@tuempresa.com",
            "subject": "Reporte Diario - {{action_1.fecha}}",
            "body": """
                Resumen del día:

                Total transacciones: {{action_1.total_transacciones}}
                Monto total: ${{action_1.monto_total}}
                Promedio: ${{action_1.monto_promedio}}

                Ver detalles en Google Sheets.
            """
        }
    }
}

print("✓ Configuración Zapier definida")
print("\nEste Zap:")
print("1. Se ejecuta diariamente a las 23:00")
print("2. Obtiene el reporte diario de la API")
print("3. Lo guarda en Google Sheets")
print("4. Envía email con resumen")

✓ Configuración Zapier definida

Este Zap:
1. Se ejecuta diariamente a las 23:00
2. Obtiene el reporte diario de la API
3. Lo guarda en Google Sheets
4. Envía email con resumen


## 6. Testing

### 6.1. Test de endpoints básicos

In [17]:
def test_api_endpoints():
    """Suite de tests para la API

    Tests incluidos:
        1. Health check
        2. Autenticación (con y sin API key)
        3. Crear cliente
        4. Crear transacción
        5. Obtener transacciones
        6. Reporte diario
    """
    BASE_URL = "http://localhost:5000/api"
    API_KEY = "api_key_demo_12345"

    headers = {
        "X-API-Key": API_KEY,
        "Content-Type": "application/json"
    }

    print("=" * 60)
    print("EJECUTANDO TESTS")
    print("=" * 60)

    # Test 1: Health check
    print("\n1. Test: Health check")
    try:
        response = requests.get(f"{BASE_URL}/health")
        assert response.status_code == 200
        assert response.json()['status'] == 'ok'
        print("   ✓ PASS")
    except Exception as e:
        print(f"   ✗ FAIL: {e}")

    # Test 2: Autenticación sin API key
    print("\n2. Test: Sin API key debe fallar (401)")
    try:
        response = requests.get(f"{BASE_URL}/customers")
        assert response.status_code == 401
        print("   ✓ PASS")
    except Exception as e:
        print(f"   ✗ FAIL: {e}")

    # Test 3: Autenticación con API key
    print("\n3. Test: Con API key debe funcionar (200)")
    try:
        response = requests.get(f"{BASE_URL}/customers", headers=headers)
        assert response.status_code == 200
        print("   ✓ PASS")
    except Exception as e:
        print(f"   ✗ FAIL: {e}")

    # Test 4: Crear cliente
    print("\n4. Test: Crear nuevo cliente")
    try:
        data = {
            "email": f"test_{datetime.now().timestamp()}@ejemplo.com",
            "nombre": "Test",
            "apellido": "Usuario",
            "pais": "ES"
        }
        response = requests.post(
            f"{BASE_URL}/customers",
            headers=headers,
            json=data
        )
        assert response.status_code == 201
        customer_id = response.json()['data']['id']
        print(f"   ✓ PASS - ID: {customer_id}")
    except Exception as e:
        print(f"   ✗ FAIL: {e}")
        customer_id = 1  # Usar ID existente si falla

    # Test 5: Crear transacción
    print("\n5. Test: Crear nueva transacción")
    try:
        data = {
            "customer_id": customer_id,
            "monto": 99.99,
            "moneda": "EUR",
            "tipo": "payment",
            "descripcion": "Test desde notebook"
        }
        response = requests.post(
            f"{BASE_URL}/transactions",
            headers=headers,
            json=data
        )
        assert response.status_code == 201
        transaction_id = response.json()['data']['id']
        print(f"   ✓ PASS - ID: {transaction_id}")
    except Exception as e:
        print(f"   ✗ FAIL: {e}")

    # Test 6: Obtener transacciones
    print("\n6. Test: Obtener transacciones")
    try:
        response = requests.get(f"{BASE_URL}/transactions", headers=headers)
        assert response.status_code == 200
        data = response.json()
        print(f"   ✓ PASS - Total: {len(data['data'])} transacciones")
    except Exception as e:
        print(f"   ✗ FAIL: {e}")

    # Test 7: Reporte diario
    print("\n7. Test: Reporte diario")
    try:
        response = requests.get(f"{BASE_URL}/reports/daily", headers=headers)
        assert response.status_code == 200
        data = response.json()
        print(f"   ✓ PASS")
        print(f"      - Transacciones: {data['total_transacciones']}")
        print(f"      - Monto total: ${data['monto_total']}")
    except Exception as e:
        print(f"   ✗ FAIL: {e}")

    print("\n" + "=" * 60)
    print("TESTS COMPLETADOS")
    print("=" * 60)

# Descomentar para ejecutar tests
# test_api_endpoints()

print("✓ Suite de tests definida")
print("\nPara ejecutar los tests:")
print("1. Asegúrate de que la API esté corriendo (python api/app.py)")
print("2. Descomenta la línea: test_api_endpoints()")
print("3. Ejecuta esta celda nuevamente")

✓ Suite de tests definida

Para ejecutar los tests:
1. Asegúrate de que la API esté corriendo (python api/app.py)
2. Descomenta la línea: test_api_endpoints()
3. Ejecuta esta celda nuevamente


In [22]:
from flask import Flask
from flask_sqlalchemy import SQLAlchemy

app = Flask(__name__)
# Configuración de la base de datos (se creará un archivo llamado indicadores.db)
app.config['SQLALCHEMY_DATABASE_URI'] = 'sqlite:///indicadores.db'
app.config['SQLALCHEMY_TRACK_MODIFICATIONS'] = False

db = SQLAlchemy(app)

# --- MODELOS PARA INDICADORES MACROECONÓMICOS ---

class Inflacion(db.Model):
    __tablename__ = 'inflacion'
    id = db.Column(db.Integer, primary_key=True)
    fecha = db.Column(db.String(10), nullable=False)
    valor = db.Column(db.Float, nullable=False)
    pais = db.Column(db.String(50), nullable=False)

    def to_dict(self):
        return {"id": self.id, "fecha": self.fecha, "valor": self.valor, "pais": self.pais}

class Desempleo(db.Model):
    __tablename__ = 'desempleo'
    id = db.Column(db.Integer, primary_key=True)
    fecha = db.Column(db.String(10), nullable=False)
    tasa = db.Column(db.Float, nullable=False)
    pais = db.Column(db.String(50), nullable=False)

    def to_dict(self):
        return {"id": self.id, "fecha": self.fecha, "tasa": self.tasa, "pais": self.pais}

class DeudaPublica(db.Model):
    __tablename__ = 'deuda_publica'
    id = db.Column(db.Integer, primary_key=True)
    fecha = db.Column(db.String(10), nullable=False)
    monto_millones = db.Column(db.Float, nullable=False)
    pais = db.Column(db.String(50), nullable=False)

    def to_dict(self):
        return {"id": self.id, "fecha": self.fecha, "monto_millones": self.monto_millones, "pais": self.pais}

# Crear las tablas en la base de datos
with app.app_context():
    db.create_all()
    print("Tablas de indicadores creadas exitosamente.")

Tablas de indicadores creadas exitosamente.
